# Check inicial — Workshop Watsonx

Verifica que todo lo necesario esté en orden **antes** de empezar a tocar credenciales o llamar a la API. No consume créditos ni hace llamadas a modelos. Es puro health check.

Si todos los checks salen `[OK]`, estás listo para el ejercicio `00_llm_conceptos`.  
Si algo sale `[FALTA]` o `[ERROR]`, resolvelo antes de avanzar.

## Helpers
Funciones auxiliares para imprimir los checks de forma consistente.

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import sys
import socket
from urllib import request, error

OK_TAG    = '[OK]   '
FALTA_TAG = '[FALTA]'
INFO_TAG  = '[--]   '

def check(label, ok, ok_detail='', fail_detail=''):
    if ok:
        line = f'  {OK_TAG} {label}'
        if ok_detail:
            line += f' — {ok_detail}'
    else:
        line = f'  {FALTA_TAG} {label}'
        if fail_detail:
            line += f' — {fail_detail}'
    print(line)
    return ok

def info(label, present):
    tag = OK_TAG if present else INFO_TAG
    print(f'  {tag} {label}')
    return present

def resolve_path(value: str) -> str:
    """Resuelve una ruta del .env relativa a la raíz del proyecto."""
    if not value:
        return value
    p = Path(value)
    if p.is_absolute():
        return str(p)
    return str(PROJECT_ROOT / p)

DOTENV_PATH  = find_dotenv(usecwd=True)
PROJECT_ROOT = Path(DOTENV_PATH).parent if DOTENV_PATH else Path.cwd()


# Acumulador del estado global
all_ok = True

## 1. Entorno Python
Verifica la versión de Python y que las librerías clave del workshop estén instaladas.

In [5]:
all_ok = check(
    f'Python {sys.version_info.major}.{sys.version_info.minor}',
    sys.version_info >= (3, 9),
    fail_detail='se requiere Python 3.9+'
) and all_ok

for lib in ['dotenv', 'ibm_watsonx_ai', 'elasticsearch']:
    try:
        __import__(lib)
        all_ok = check(f'import {lib}', True) and all_ok
    except ImportError as e:
        all_ok = check(f'import {lib}', False, fail_detail=str(e)) and all_ok

  [OK]    Python 3.11
  [OK]    import dotenv
  [OK]    import ibm_watsonx_ai
  [OK]    import elasticsearch


## 2. Archivo .env
Verifica que el archivo `.env` exista y se pueda cargar.

In [6]:
from dotenv import load_dotenv
 
env_loaded = load_dotenv(DOTENV_PATH) if DOTENV_PATH else False
all_ok = check(
    '.env encontrado y cargado',
    env_loaded,
    ok_detail=f'desde {DOTENV_PATH}' if env_loaded else '',
    fail_detail='ejecutá `cp example.txt .env` y completá los valores'
) and all_ok

  [OK]    .env encontrado y cargado — desde /app/.env


## 3. Variables watsonx.ai (obligatorias)
Estas variables son **obligatorias** para el onboarding del Día 1. Si alguna falta, no vas a poder correr el ejercicio 01.

In [7]:
wx_vars = ['WX_API_KEY', 'WX_PROJECT_ID', 'WX_URL']
for var in wx_vars:
    val = os.getenv(var, '')
    present = bool(val) and 'TU_' not in val
    preview = f'{val[:8]}...' if present and len(val) > 8 else val
    all_ok = check(
        var, present,
        ok_detail=preview,
        fail_detail='vacío o placeholder'
    ) and all_ok

  [OK]    WX_API_KEY — ZqQTjHQn...
  [OK]    WX_PROJECT_ID — 8cef60a9...
  [OK]    WX_URL — https://...


## 4. Variables Elasticsearch (opcional, Día 3)
Estas variables **no son bloqueantes hoy**, pero las vas a necesitar para el ejercicio del Día 3. Si no las tenés todavía, no es problema.

In [8]:
es_vars = ['ES_URL', 'ES_USER', 'ES_PASSWORD', 'ES_INDEX', 'ES_CA_CERT']
for var in es_vars:
    val = os.getenv(var, '')
    present = bool(val) and 'TU_' not in val
    info(var, present)
 
# Verificar que el cert exista solo si está configurado (no bloqueante)
ca_cert_raw = os.getenv('ES_CA_CERT', '')
if ca_cert_raw:
    ca_cert_resolved = resolve_path(ca_cert_raw)
    cert_exists = os.path.isfile(ca_cert_resolved)
    info(f'archivo de cert en {ca_cert_resolved}', cert_exists)
    if not cert_exists:
        print('          (no bloqueante hoy, pero lo vas a necesitar en el Día 3)')

  [OK]    ES_URL
  [OK]    ES_USER
  [OK]    ES_PASSWORD
  [OK]    ES_INDEX
  [OK]    ES_CA_CERT
  [OK]    archivo de cert en /app/certs/ca.crt


## 5. Conectividad a IBM Cloud
Verifica que el contenedor pueda resolver DNS y hacer HTTPS contra IBM Cloud. Si esto falla, lo más probable es que sea un problema de red corporativa o proxy.

In [9]:
try:
    socket.gethostbyname('iam.cloud.ibm.com')
    all_ok = check('DNS resuelve iam.cloud.ibm.com', True) and all_ok
except socket.gaierror as e:
    all_ok = check('DNS resuelve iam.cloud.ibm.com', False, fail_detail=str(e)) and all_ok

try:
    req = request.Request('https://iam.cloud.ibm.com/identity/.well-known/openid-configuration')
    with request.urlopen(req, timeout=5) as resp:
        all_ok = check(
            'HTTPS a iam.cloud.ibm.com',
            resp.status == 200,
            ok_detail=f'status {resp.status}',
            fail_detail=f'status inesperado {resp.status}'
        ) and all_ok
except (error.URLError, TimeoutError) as e:
    all_ok = check(
        'HTTPS a iam.cloud.ibm.com',
        False,
        fail_detail=f'problema de red o proxy: {e}'
    ) and all_ok

  [OK]    DNS resuelve iam.cloud.ibm.com
  [OK]    HTTPS a iam.cloud.ibm.com — status 200


## Resultado final

In [10]:
print('=' * 50)
if all_ok:
    print(' Todo OK. Podés avanzar al ejercicio 00_llm_conceptos.')
else:
    print(' Hay items que resolver. Revisá los [FALTA] y [ERROR] arriba.')
    print(' Lo más común: completar el .env o instalar dependencias.')
print('=' * 50)

 Todo OK. Podés avanzar al ejercicio 00_llm_conceptos.
